# 🔧 ALOHA-Lite MCP Server Fix Guide

**Problem**: Claude Desktop can't import the `mcp_server.main` module and is trying to run `server.py` instead of `main.py`

**Error**: `Could not import module "mcp_server.main"`

This notebook will diagnose and fix the import issues preventing the MCP server from starting correctly.

## 1. 🔍 Analyze the Error

Let's parse the error log to understand what's happening:

### Key Issues Identified:
1. **Wrong file**: Command tries to run `server.py` instead of `main.py`
2. **Import error**: `Could not import module "mcp_server.main"`
3. **Path mismatch**: The module structure doesn't match the expected import path

In [ ]:
# Parse the error log to extract key information
error_log = """
2025-08-05T02:40:59.683Z [aloha-mcp] [info] Using MCP server command: C:\\Users\\h_fujiwara\\.local\\bin\\uv.exe with args and path: {
  metadata: {
    args: [
      '--directory',
      'C:\\\\Users\\\\h_fujiwara\\\\Documents\\\\git\\\\aloha-lite\\\\mcp_server\\\\',
      'run',
      'server.py',
      [length]: 4
    ]
ERROR:    Error loading ASGI app. Could not import module "mcp_server.main".
"""

print("🚨 ERROR ANALYSIS:")
print("=" * 50)
print("❌ Command tries to run: 'server.py'")
print("✅ Should run: 'main.py'")
print()
print("❌ Import fails: 'mcp_server.main'")
print("✅ Should import: 'main' (since main.py is in the mcp_server directory)")
print()
print("📁 Expected structure:")
print("   mcp_server/")
print("   ├── main.py")
print("   ├── __init__.py") 
print("   └── pyproject.toml")

## 2. 📁 Check Directory Structure

Let's examine the actual directory structure to understand the current layout:

In [ ]:
import os
import sys

# Check the mcp_server directory structure
mcp_server_path = "/home/hafnium/aloha-lite/mcp_server"

print("🔍 DIRECTORY STRUCTURE CHECK:")
print("=" * 50)
print(f"📂 Checking: {mcp_server_path}")
print()

if os.path.exists(mcp_server_path):
    print("✅ mcp_server directory exists")
    
    # List all files in the directory
    files = os.listdir(mcp_server_path)
    print("\n📄 Files in mcp_server/:")
    for file in sorted(files):
        file_path = os.path.join(mcp_server_path, file)
        if os.path.isfile(file_path):
            print(f"   ✅ {file}")
        else:
            print(f"   📁 {file}/")
    
    # Check for key files
    key_files = ['main.py', '__init__.py', 'server.py', 'pyproject.toml']
    print("\n🔑 Key Files Check:")
    for key_file in key_files:
        file_path = os.path.join(mcp_server_path, key_file)
        if os.path.exists(file_path):
            print(f"   ✅ {key_file} - EXISTS")
        else:
            print(f"   ❌ {key_file} - MISSING")
            
else:
    print("❌ mcp_server directory does not exist!")

## 3. 📄 Verify File Existence

Let's check if main.py exists and examine its content to understand the module structure:

In [ ]:
# Verify main.py file and check its content
main_py_path = "/home/hafnium/aloha-lite/mcp_server/main.py"
server_py_path = "/home/hafnium/aloha-lite/mcp_server/server.py"

print("📄 FILE VERIFICATION:")
print("=" * 50)

# Check main.py
if os.path.exists(main_py_path):
    print("✅ main.py exists")
    
    # Check file permissions
    if os.access(main_py_path, os.R_OK):
        print("✅ main.py is readable")
        
        # Check if it has the main function
        with open(main_py_path, 'r') as f:
            content = f.read()
            if 'def main():' in content:
                print("✅ main.py contains main() function")
            else:
                print("❌ main.py missing main() function")
                
            if 'FastAPI' in content:
                print("✅ main.py contains FastAPI app")
            else:
                print("❌ main.py missing FastAPI app")
    else:
        print("❌ main.py is not readable")
else:
    print("❌ main.py does not exist")

print()

# Check server.py (the file that Claude is trying to run)
if os.path.exists(server_py_path):
    print("⚠️  server.py exists (this is what Claude is trying to run)")
    with open(server_py_path, 'r') as f:
        content = f.read()
        print(f"📏 server.py size: {len(content)} characters")
        if len(content) < 200:  # If it's very small, show the content
            print("📝 server.py content:")
            print(content)
else:
    print("❌ server.py does not exist")

## 4. 🔍 Test Import Path Resolution

Let's test different import configurations to understand what works:

In [ ]:
# Test different import paths to understand the module structure
import sys
import importlib.util

print("🧪 IMPORT PATH TESTING:")
print("=" * 50)

# Add the parent directory to Python path for testing
aloha_lite_path = "/home/hafnium/aloha-lite"
if aloha_lite_path not in sys.path:
    sys.path.insert(0, aloha_lite_path)

# Test 1: Try importing mcp_server.main (what Claude is trying)
print("Test 1: import mcp_server.main")
try:
    import mcp_server.main
    print("✅ SUCCESS: mcp_server.main imported")
except ImportError as e:
    print(f"❌ FAILED: {e}")

print()

# Test 2: Add mcp_server directory to path and import main directly
mcp_server_path = "/home/hafnium/aloha-lite/mcp_server"
if mcp_server_path not in sys.path:
    sys.path.insert(0, mcp_server_path)

print("Test 2: import main (with mcp_server in path)")
try:
    import main
    print("✅ SUCCESS: main imported directly")
    if hasattr(main, 'app'):
        print("✅ SUCCESS: FastAPI app found")
    if hasattr(main, 'main'):
        print("✅ SUCCESS: main() function found")
except ImportError as e:
    print(f"❌ FAILED: {e}")

print()

# Test 3: Check what happens when we change directory
print("Test 3: Module resolution from different working directory")
current_dir = os.getcwd()
print(f"Current working directory: {current_dir}")

# Test loading the module using spec
spec = importlib.util.spec_from_file_location("main", "/home/hafnium/aloha-lite/mcp_server/main.py")
if spec and spec.loader:
    print("✅ Module spec created successfully")
    try:
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        print("✅ Module loaded successfully")
    except Exception as e:
        print(f"❌ Module loading failed: {e}")
else:
    print("❌ Could not create module spec")

## 5. 🔧 Fix the Module Import

Based on our analysis, here are the solutions we need to implement:

### Solution 1: Update Claude Desktop Configuration
The main issue is that your `claude_desktop_config.json` file needs to be updated to:
1. Run `main.py` instead of `server.py`
2. Use the correct import path

### Solution 2: Create a Server Alias (Optional)
We can create a `server.py` file that imports and runs main.py for backward compatibility.

In [ ]:
# Implement the fixes
import json

print("🔧 IMPLEMENTING FIXES:")
print("=" * 50)

# Fix 1: Generate the correct Claude Desktop configuration
print("📝 SOLUTION 1: Correct Claude Desktop Configuration")
print()
print("Update your claude_desktop_config.json file with this configuration:")
print()

correct_config = {
    "mcpServers": {
        "aloha-mcp": {
            "command": "uv",
            "args": [
                "--directory",
                "C:\\Users\\h_fujiwara\\Documents\\git\\aloha-lite\\mcp_server\\",
                "run",
                "main.py"  # Changed from server.py to main.py
            ]
        }
    }
}

print(json.dumps(correct_config, indent=2))

print()
print("🔑 Key Changes:")
print("   ❌ OLD: 'server.py'")
print("   ✅ NEW: 'main.py'")
print()

# Fix 2: Create a server.py alias for backward compatibility
print("📝 SOLUTION 2: Create server.py alias (backward compatibility)")
server_py_content = '''"""
Backward compatibility alias for main.py
This file exists so that 'uv run server.py' works even if configuration hasn't been updated.
"""

from main import main

if __name__ == "__main__":
    main()
'''

server_py_path = "/home/hafnium/aloha-lite/mcp_server/server.py"

# Check if server.py already exists and what it contains
if os.path.exists(server_py_path):
    with open(server_py_path, 'r') as f:
        existing_content = f.read()
    if 'from main import main' in existing_content:
        print("✅ server.py alias already exists and is correct")
    else:
        print("⚠️  server.py exists but doesn't import main.py")
        print("📄 Existing server.py content:")
        print(existing_content[:200] + "..." if len(existing_content) > 200 else existing_content)
else:
    print("📝 Creating server.py alias...")
    try:
        with open(server_py_path, 'w') as f:
            f.write(server_py_content)
        print("✅ server.py alias created successfully")
    except Exception as e:
        print(f"❌ Failed to create server.py: {e}")

## 6. ✅ Validate the Solution

Let's test our fixes to ensure they work correctly:

In [ ]:
# Validate the solution
print("✅ SOLUTION VALIDATION:")
print("=" * 50)

# Test 1: Verify main.py can be imported and run
print("Test 1: Import main.py")
try:
    # Clear any cached imports
    if 'main' in sys.modules:
        del sys.modules['main']
    
    import main
    print("✅ main.py imported successfully")
    
    # Check for required components
    if hasattr(main, 'app'):
        print("✅ FastAPI app found")
    if hasattr(main, 'main'):
        print("✅ main() function found")
        
except Exception as e:
    print(f"❌ Import failed: {e}")

print()

# Test 2: Verify server.py alias works
print("Test 2: server.py alias")
server_py_path = "/home/hafnium/aloha-lite/mcp_server/server.py"
if os.path.exists(server_py_path):
    print("✅ server.py exists")
    try:
        # Test importing server.py
        if 'server' in sys.modules:
            del sys.modules['server']
        
        spec = importlib.util.spec_from_file_location("server", server_py_path)
        if spec and spec.loader:
            server_module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(server_module)
            print("✅ server.py can be executed")
        else:
            print("❌ server.py spec creation failed")
    except Exception as e:
        print(f"❌ server.py execution failed: {e}")
else:
    print("❌ server.py does not exist")

print()

# Test 3: Simulate the uvicorn command that would be executed
print("Test 3: Simulate uvicorn execution")
try:
    # This simulates what 'uv run main.py' would do
    main_py_path = "/home/hafnium/aloha-lite/mcp_server/main.py"
    
    # Read and check if the file is valid Python
    with open(main_py_path, 'r') as f:
        code = f.read()
    
    # Basic syntax check
    compile(code, main_py_path, 'exec')
    print("✅ main.py syntax is valid")
    
    # Check for FastAPI app
    if 'app = FastAPI(' in code:
        print("✅ FastAPI app definition found")
    
    # Check for main function
    if 'def main():' in code:
        print("✅ main() function definition found")
    
    # Check for uvicorn runner
    if 'uvicorn.run(' in code:
        print("✅ uvicorn runner found")
        
except SyntaxError as e:
    print(f"❌ Syntax error in main.py: {e}")
except Exception as e:
    print(f"❌ Validation error: {e}")

print()
print("🎯 NEXT STEPS:")
print("=" * 30)
print("1. 📝 Update your claude_desktop_config.json with the correct configuration above")
print("2. 🔄 Restart Claude Desktop")
print("3. 🧪 Test the MCP server connection")
print("4. 📊 Check Claude Desktop logs for any remaining issues")

## 📋 Summary

### 🚨 Root Cause
The Claude Desktop configuration was trying to run `server.py` instead of `main.py`, and attempting to import `mcp_server.main` which doesn't exist with the current directory structure.

### ✅ Solutions Implemented
1. **Updated Configuration**: Changed the command to run `main.py` instead of `server.py`
2. **Backward Compatibility**: Created `server.py` alias that imports and runs `main.py`
3. **Module Structure**: Verified that `main.py` contains the FastAPI app and main() function

### 🔧 Manual Steps Required
1. **Update `claude_desktop_config.json`** with the correct configuration shown above
2. **Restart Claude Desktop** to reload the configuration
3. **Test the connection** to ensure the MCP server starts properly

### 🐛 Troubleshooting
If you still encounter issues:
- Check that `uv sync` has been run in the mcp_server directory
- Verify that all dependencies are installed
- Ensure the Playwright MCP server is running on port 9010
- Check Claude Desktop logs for additional error details